<a href="https://colab.research.google.com/github/fjnarvaezo1-dotcom/Lineal-regresion-machine-learning/blob/main/proyecto_zendesk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TEXT FOR EXPORT DATA


In [6]:
import requests
import json
import pandas as pd


def login():
    url = "https://yamahamotor-financiera.zendesk.com/oauth/tokens"
    payload = {
    "grant_type": "client_credentials",
    "client_id": "validacion_informacion_apis",
    "client_secret": "a522da046ff85a6bbf8a0f2d9374e0f5cc74c7dfe710683475b7b42c64956b87",
    "scope": "tickets:read"
}

    headers = {
    "Content-Type": "application/json"
}
    response = requests.post(url, json=payload, headers=headers)
    data = response.json()
    access_token = data["access_token"]
    return access_token

a = login()
url = "https://yamahamotor-financiera.zendesk.com/api/v2/tickets"
headers = {"Authorization": "Bearer {}".format(a)}
response = requests.get(url=url,headers=headers)
data = response.json()
dataframe  = pd.DataFrame(data)
print(dataframe)











                                              tickets  \
0   {'url': 'https://yamahamotor-financiera.zendes...   
1   {'url': 'https://yamahamotor-financiera.zendes...   
2   {'url': 'https://yamahamotor-financiera.zendes...   
3   {'url': 'https://yamahamotor-financiera.zendes...   
4   {'url': 'https://yamahamotor-financiera.zendes...   
..                                                ...   
95  {'url': 'https://yamahamotor-financiera.zendes...   
96  {'url': 'https://yamahamotor-financiera.zendes...   
97  {'url': 'https://yamahamotor-financiera.zendes...   
98  {'url': 'https://yamahamotor-financiera.zendes...   
99  {'url': 'https://yamahamotor-financiera.zendes...   

                                            next_page previous_page  count  
0   https://yamahamotor-financiera.zendesk.com/api...          None    181  
1   https://yamahamotor-financiera.zendesk.com/api...          None    181  
2   https://yamahamotor-financiera.zendesk.com/api...          None    181  
3   htt

In [7]:
for i,a in dataframe.items():
  if i == "tickets":
    for a in a :
      print(a)

{'url': 'https://yamahamotor-financiera.zendesk.com/api/v2/tickets/3.json', 'id': 3, 'external_id': None, 'via': {'channel': 'api', 'source': {'from': {}, 'to': {}, 'rel': None}}, 'created_at': '2026-09-17T13:56:59Z', 'updated_at': '2026-09-17T14:05:55Z', 'generated_timestamp': 1789653955, 'type': 'incident', 'subject': 'Llamada registrada desde Wolkvox - 1234567 ', 'raw_subject': 'Llamada registrada desde Wolkvox - 1234567 ', 'description': 'pruebas', 'priority': 'normal', 'status': 'solved', 'recipient': None, 'requester_id': 53660303838100, 'submitter_id': 53660303838100, 'assignee_id': 53145562564244, 'organization_id': None, 'group_id': 53145563736980, 'collaborator_ids': [], 'follower_ids': [], 'email_cc_ids': [], 'forum_topic_id': None, 'problem_id': None, 'has_incidents': False, 'is_public': True, 'due_at': None, 'tags': ['9', 'abejorral', 'antioquia', 'cliente_actual', 'cliente_no_escucha', 'informacion', 'levantamiento_de_prenda', 'libranza', 'linea_telefonica', 'marilyn_arte

### ESTRAER INFORMCION DE LOS CASOS 1202 DE MOMENTO

In [9]:


def informacion(url):
    a = login()
    headers = {"Authorization": "Bearer {}".format(a)}
    response = requests.get(url=url, headers=headers)
    data = response.json()
    return data


# Llamada a la función pasando la URL como argumento:
endpoint = "https://yamahamotor-financiera.zendesk.com/api/v2/tickets/81.json"
resultado = informacion(endpoint)
print(resultado)






{'ticket': {'url': 'https://yamahamotor-financiera.zendesk.com/api/v2/tickets/81.json', 'id': 81, 'external_id': None, 'via': {'channel': 'api', 'source': {'from': {}, 'to': {}, 'rel': None}}, 'created_at': '2026-09-18T13:20:36Z', 'updated_at': '2026-09-18T13:20:36Z', 'generated_timestamp': 1789737637, 'type': None, 'subject': 'Llamada registrada desde Wolkvox - inbound ', 'raw_subject': 'Llamada registrada desde Wolkvox - inbound ', 'description': 'Llamada registrada en Wolkvox. Cola: 4158 - CALI(4158) - Tel: 3122889599 ID_CALL: 0000.0760.8673.1789737609.403888 - inbound - 2026-9-18 08:20:36 - Login : 12533', 'priority': 'normal', 'status': 'new', 'recipient': None, 'requester_id': 53663370039700, 'submitter_id': 53663370039700, 'assignee_id': None, 'organization_id': None, 'group_id': 53145563736980, 'collaborator_ids': [], 'follower_ids': [], 'email_cc_ids': [], 'forum_topic_id': None, 'problem_id': None, 'has_incidents': False, 'is_public': True, 'due_at': None, 'tags': [], 'custom

########### esporte de datos incolmotos

In [ ]:
#!/usr/bin/env python3
"""
Migra tickets (con comentarios y adjuntos) de
    incolmotosyamaha.zendesk.com  ->  yamahamotor-financiera.zendesk.com

Usa la API de Ticket Import (/api/v2/imports/tickets), que conserva fechas
originales, permite tickets cerrados y NO dispara triggers/notificaciones.

Los IDs (usuarios, grupos, campos, formularios, marcas) son distintos en cada
subdominio, por eso el script los "traduce" por email / nombre / título.

Los tickets a migrar se leen de un CSV (columna "url", separador ";").

Uso:
    python migrar_tickets_zendesk.py --dry-run --limit 5     # prueba sin crear
    python migrar_tickets_zendesk.py --ticket 1101682        # un solo ticket
    python migrar_tickets_zendesk.py                         # todos los del CSV
    python migrar_tickets_zendesk.py --csv otro.csv --limit 50
    python migrar_tickets_zendesk.py --all                   # todo el origen (sin CSV)

AUTENTICACIÓN  ->  ver la sección "AUTENTICACIÓN" más abajo (una sola zona):
    ORIGEN  (incolmotosyamaha)       : header fijo  Authorization: Basic xxx
    DESTINO (yamahamotor-financiera) : OAuth client_credentials -> login() -> Bearer
"""
import argparse
import json
import logging
import os
import sys
import time
from datetime import datetime, timezone

import re

import pandas as pd
import requests

# CSV con las URLs de los tickets del ORIGEN (columna "url", separador ";")
CSV_PATH = r"C:\Users\fnarvaez\OneDrive - WALTER BRIDGE Y CIA SA - BRIDGE SA\Escritorio\tickets.csv"
CSV_SEP = ";"
CSV_URL_COLUMN = "url"

SRC = "https://incolmotosyamaha.zendesk.com"
DST = "https://yamahamotor-financiera.zendesk.com"

CHECKPOINT_FILE = "migracion_checkpoint.json"   # {"src_id": dst_id}
ERRORS_FILE = "migracion_errores.jsonl"
MIGRATION_TAG = "migrado_incolmotos"
SKIP_ATTACHMENTS = False

# Agente destino a usar cuando el autor/asignado original no exista en destino
# (ID numérico del agente en yamahamotor-financiera). Opcional pero recomendado.
DEFAULT_AGENT_ID = int(os.getenv("DST_DEFAULT_AGENT_ID", "0")) or None

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("migracion")


# ==========================================================================
# AUTENTICACIÓN  (TODO lo relacionado con credenciales está en esta sección)
# ==========================================================================

# ---- ORIGEN: incolmotosyamaha.zendesk.com  ->  Authorization: Basic xxx ----
# Pega aquí tu valor (recomendado: mejor en variable de entorno SRC_BASIC).
SRC_BASIC_AUTH = os.getenv("SRC_BASIC", "Basic xxxxx")


def auth_header_origen():
    """Header de autenticación del subdominio ORIGEN (incolmotosyamaha)."""
    v = SRC_BASIC_AUTH.strip()
    if not v.lower().startswith("basic "):
        v = f"Basic {v}"
    return v


# ---- DESTINO: yamahamotor-financiera.zendesk.com -> OAuth client_credentials ----
DST_CLIENT_ID = os.getenv("DST_CLIENT_ID", "validacion_informacion_apis")
DST_CLIENT_SECRET = os.getenv("DST_CLIENT_SECRET", "xxxxxx")
# OJO: con "tickets:read" NO se puede crear tickets. Para migrar se necesita escritura.
# "read write" cubre tickets, imports, usuarios, organizaciones y uploads.
DST_SCOPE = os.getenv("DST_SCOPE", "tickets:read tickets:write")


def login():
    """Tu login(): pide el access_token del DESTINO (yamahamotor-financiera)."""
    url = f"{DST}/oauth/tokens"
    if DST_CLIENT_SECRET.strip().lower() in ("", "xxxxxx", "xxxxx"):
        sys.exit("login(): DST_CLIENT_SECRET sigue con el valor de ejemplo. "
                 "Define la variable de entorno DST_CLIENT_SECRET (misma terminal donde corres el script) "
                 "o pega el secret real en la constante.")
    payload = {
        "grant_type": "client_credentials",
        "client_id": DST_CLIENT_ID,
        "client_secret": DST_CLIENT_SECRET,
        "scope": DST_SCOPE,
    }
    headers = {"Content-Type": "application/json"}
    response = requests.post(url, json=payload, headers=headers, timeout=60)
    data = response.json()
    if "access_token" not in data:
        sys.exit(f"login() falló ({response.status_code}): {data}")
    return data["access_token"]


def auth_header_destino():
    """Header de autenticación del subdominio DESTINO.
    - Si defines DST_BASIC (email/token de API en base64): usa Basic.
    - Si no: usa OAuth client_credentials -> login() -> Bearer."""
    basic = os.getenv("DST_BASIC", "").strip()
    if basic:
        return basic if basic.lower().startswith("basic ") else f"Basic {basic}"
    return f"Bearer {login()}"


# ==========================================================================
# Cliente HTTP con reintentos, rate limit (429) y renovación de token (401)
# ==========================================================================
class Zendesk:
    def __init__(self, base, auth_header_fn):
        self.base = base
        self.auth_header_fn = auth_header_fn
        self.s = requests.Session()
        self.refresh_auth()

    def refresh_auth(self):
        self.s.headers["Authorization"] = self.auth_header_fn()

    def request(self, method, path_or_url, retries=6, **kw):
        url = path_or_url if path_or_url.startswith("http") else self.base + path_or_url
        for attempt in range(retries):
            try:
                r = self.s.request(method, url, timeout=60, **kw)
            except requests.RequestException as e:
                log.warning("Error de red (%s), reintentando...", e)
                time.sleep(2 ** attempt)
                continue
            if r.status_code == 401 and attempt == 0 and self.base == DST:
                log.warning("401 en destino: renovando token con login()")
                self.refresh_auth()
                continue
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 30))
                log.warning("429 rate limit, esperando %ss", wait)
                time.sleep(wait + 1)
                continue
            if r.status_code >= 500:
                time.sleep(2 ** attempt)
                continue
            return r
        raise RuntimeError(f"Falló {method} {url} tras {retries} intentos")

    def get(self, path, **kw):
        r = self.request("GET", path, **kw)
        r.raise_for_status()
        return r.json()

    def post(self, path, **kw):
        return self.request("POST", path, **kw)

    def paginate(self, path, key, params=None):
        """Paginación por cursor genérica (page[size])."""
        params = dict(params or {})
        params["page[size]"] = 100
        url, first = path, True
        while url:
            data = self.get(url, params=params if first else None)
            first = False
            yield from data.get(key, [])
            url = data.get("links", {}).get("next") if data.get("meta", {}).get("has_more") else None


# --------------------------------------------------------------------------
# Mapeos entre cuentas (por nombre / título / email)
# --------------------------------------------------------------------------
class Mapper:
    def __init__(self, src: Zendesk, dst: Zendesk):
        self.src, self.dst = src, dst
        self.user_cache = {}      # src_user_id -> dst_user_id
        self.org_cache = {}
        self._build_static_maps()

    def _by_name(self, key, name_attr="name"):
        s = {x["id"]: x for x in self.src.paginate(f"/api/v2/{key}.json", key)}
        d = {x[name_attr].strip().lower(): x["id"] for x in self.dst.paginate(f"/api/v2/{key}.json", key)}
        return {sid: d.get(x[name_attr].strip().lower()) for sid, x in s.items()}

    def _build_static_maps(self):
        log.info("Construyendo mapas de grupos, formularios, marcas y campos...")
        self.groups = self._by_name("groups")
        self.brands = self._by_name("brands")
        try:
            forms_s = self.src.get("/api/v2/ticket_forms.json")["ticket_forms"]
            forms_d = {f["name"].strip().lower(): f["id"] for f in self.dst.get("/api/v2/ticket_forms.json")["ticket_forms"]}
            self.forms = {f["id"]: forms_d.get(f["name"].strip().lower()) for f in forms_s}
        except Exception as e:
            log.warning("No se pudieron mapear formularios: %s", e)
            self.forms = {}

        # Campos personalizados: se emparejan por título y tipo
        fs = self.src.get("/api/v2/ticket_fields.json?page[size]=100")["ticket_fields"]
        fd = self.dst.get("/api/v2/ticket_fields.json?page[size]=100")["ticket_fields"]
        d_index = {(f["title"].strip().lower(), f["type"]): f["id"] for f in fd if f.get("active", True)}
        self.fields, self.unmapped_fields = {}, []
        for f in fs:
            if f["type"] in ("subject", "description", "status", "tickettype", "priority",
                             "group", "assignee", "custom_status"):
                continue  # campos del sistema
            did = d_index.get((f["title"].strip().lower(), f["type"]))
            if did:
                self.fields[f["id"]] = did
            else:
                self.unmapped_fields.append(f"{f['id']} - {f['title']}")
        if self.unmapped_fields:
            log.warning("Campos SIN equivalente en destino (se omiten): %s", self.unmapped_fields)

    # ---- usuarios -------------------------------------------------------
    def user(self, src_uid, allow_create_end_user=True):
        """Devuelve el id del usuario en destino (o None)."""
        if not src_uid:
            return None
        if src_uid in self.user_cache:
            return self.user_cache[src_uid]
        u = self.src.get(f"/api/v2/users/{src_uid}.json")["user"]
        dst_id = None
        if u["role"] == "end-user":
            if allow_create_end_user:
                payload = {"name": u["name"] or u.get("email") or f"Usuario {src_uid}", "role": "end-user"}
                if u.get("email"):
                    payload["email"] = u["email"]
                if u.get("phone"):
                    payload["phone"] = u["phone"]
                if not u.get("email"):
                    payload["external_id"] = f"incolmotos-user-{src_uid}"
                r = self.dst.post("/api/v2/users/create_or_update.json", json={"user": payload})
                if r.ok:
                    dst_id = r.json()["user"]["id"]
                else:
                    log.warning("No se pudo crear usuario %s: %s", src_uid, r.text[:200])
        else:  # agentes/admins: NO se crean (consumen licencia); se buscan por email
            if u.get("email"):
                res = self.dst.get("/api/v2/users/search.json", params={"query": f"email:{u['email']}"})
                for cand in res.get("users", []):
                    if cand["role"] in ("agent", "admin"):
                        dst_id = cand["id"]
                        break
        self.user_cache[src_uid] = dst_id
        return dst_id

    def org(self, src_org_id):
        if not src_org_id:
            return None
        if src_org_id in self.org_cache:
            return self.org_cache[src_org_id]
        o = self.src.get(f"/api/v2/organizations/{src_org_id}.json")["organization"]
        res = self.dst.get("/api/v2/organizations/autocomplete.json", params={"name": o["name"]})
        dst_id = next((x["id"] for x in res.get("organizations", [])
                       if x["name"].strip().lower() == o["name"].strip().lower()), None)
        if not dst_id:
            r = self.dst.post("/api/v2/organizations.json", json={"organization": {"name": o["name"]}})
            dst_id = r.json()["organization"]["id"] if r.ok else None
        self.org_cache[src_org_id] = dst_id
        return dst_id


# --------------------------------------------------------------------------
# Migración de un ticket
# --------------------------------------------------------------------------
def upload_attachment(src, dst, att):
    """Descarga un adjunto del origen y lo sube al destino; devuelve token."""
    r = src.request("GET", att["content_url"])
    if not r.ok:
        log.warning("No se pudo descargar adjunto %s", att["file_name"])
        return None
    up = dst.post("/api/v2/uploads.json", params={"filename": att["file_name"]},
                  headers={"Content-Type": att.get("content_type") or "application/octet-stream"},
                  data=r.content)
    if not up.ok:
        log.warning("No se pudo subir adjunto %s: %s", att["file_name"], up.text[:200])
        return None
    return up.json()["upload"]["token"]


def build_payload(src, dst, m: Mapper, t: dict) -> dict:
    requester_id = m.user(t["requester_id"])
    if not requester_id:
        raise RuntimeError(f"Sin requester en destino para ticket {t['id']}")
    assignee_id = m.user(t.get("assignee_id"), allow_create_end_user=False) or None

    # ---- comentarios --------------------------------------------------
    comments = []
    for c in src.paginate(f"/api/v2/tickets/{t['id']}/comments.json", "comments"):
        author = m.user(c["author_id"]) or DEFAULT_AGENT_ID or requester_id
        body = c.get("html_body") or c.get("body") or "(sin contenido)"
        # Conserva autor original como nota si no se pudo mapear
        entry = {"author_id": author, "html_body": body, "public": c.get("public", True),
                 "created_at": c["created_at"]}
        tokens, failed = [], []
        for att in c.get("attachments", []):
            if SKIP_ATTACHMENTS:
                failed.append(att["file_name"] + " (omitido)")
                continue
            tok = upload_attachment(src, dst, att)
            if tok:
                tokens.append(tok)
            else:
                failed.append(att["file_name"])
        if tokens:
            entry["uploads"] = tokens
        if failed:
            entry["html_body"] += ("<hr><p><i>[Migración] Adjuntos no migrados: "
                                   + ", ".join(failed) + "</i></p>")
        comments.append(entry)
    if not comments:  # el import exige al menos un comentario
        comments.append({"author_id": requester_id, "body": t.get("description") or "(sin descripción)",
                         "created_at": t["created_at"]})

    # ---- campos personalizados ----------------------------------------
    custom_fields = []
    for f in t.get("custom_fields", []):
        did = m.fields.get(f["id"])
        if did and f["value"] not in (None, "", []):
            custom_fields.append({"id": did, "value": f["value"]})

    tags = sorted(set(t.get("tags", [])) | {MIGRATION_TAG})
    payload = {
        "subject": t.get("subject") or "(sin asunto)",
        "requester_id": requester_id,
        "status": t["status"] if t["status"] in ("new", "open", "pending", "hold", "solved", "closed") else "open",
        "created_at": t["created_at"],
        "updated_at": t["updated_at"],
        "external_id": f"incolmotos-{t['id']}",   # trazabilidad al ticket original
        "tags": tags,
        "comments": comments,
        "custom_fields": custom_fields,
    }
    if t["status"] in ("solved", "closed"):
        payload["solved_at"] = t["updated_at"]
    if t.get("priority"):
        payload["priority"] = t["priority"]
    if t.get("type"):
        payload["type"] = t["type"]
    if assignee_id:
        payload["assignee_id"] = assignee_id
    gid = m.groups.get(t.get("group_id"))
    if gid:
        payload["group_id"] = gid
    oid = m.org(t.get("organization_id"))
    if oid:
        payload["organization_id"] = oid
    fid = m.forms.get(t.get("ticket_form_id"))
    if fid:
        payload["ticket_form_id"] = fid
    bid = m.brands.get(t.get("brand_id"))
    if bid:
        payload["brand_id"] = bid
    return {"ticket": payload}


def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {}


def save_checkpoint(cp):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(cp, f)


def ids_from_csv(path, sep, column):
    """Lee el CSV y extrae los IDs de ticket desde las URLs
    (ej. https://incolmotosyamaha.zendesk.com/api/v2/tickets/1101682.json)."""
    data = pd.read_csv(path, sep=sep, dtype=str)
    if column not in data.columns:
        sys.exit(f"El CSV no tiene la columna '{column}'. Columnas: {list(data.columns)}")
    ids, bad = [], []
    for u in data[column].dropna():
        mt = re.search(r"/tickets/(\d+)", str(u))
        if mt:
            ids.append(int(mt.group(1)))
        else:
            bad.append(u)
    if bad:
        log.warning("%s URLs no tienen ID de ticket válido (ej: %s)", len(bad), bad[:3])
    ids = list(dict.fromkeys(ids))  # quita duplicados conservando el orden
    log.info("CSV: %s tickets únicos a migrar", len(ids))
    return ids


def iter_tickets_by_ids(src: Zendesk, ids):
    """Trae los tickets del origen en lotes de 100 (show_many).
    Los que no vengan en el lote (p. ej. archivados) se piden uno a uno."""
    for i in range(0, len(ids), 100):
        chunk = ids[i:i + 100]
        r = src.request("GET", "/api/v2/tickets/show_many.json",
                        params={"ids": ",".join(map(str, chunk))})
        found = {}
        if r.ok:
            found = {t["id"]: t for t in r.json().get("tickets", [])}
        for tid in chunk:
            if tid in found:
                yield found[tid]
                continue
            rr = src.request("GET", f"/api/v2/tickets/{tid}.json")
            if rr.ok:
                yield rr.json()["ticket"]
            else:
                log.error("Ticket %s no se pudo leer en origen (%s)", tid, rr.status_code)
                with open(ERRORS_FILE, "a") as f:
                    f.write(json.dumps({"src_id": tid, "error": f"lectura origen {rr.status_code}"}) + "\n")


def iter_source_tickets(src: Zendesk, start_time: int):
    """Export incremental por cursor: incluye tickets archivados/cerrados."""
    params = {"start_time": start_time}
    while True:
        data = src.get("/api/v2/incremental/tickets/cursor.json", params=params)
        for t in data["tickets"]:
            yield t
        if data.get("end_of_stream"):
            return
        params = {"cursor": data["after_cursor"]}
        time.sleep(7)  # el endpoint permite ~10 req/min


def main():
    ap = argparse.ArgumentParser()
    global SKIP_ATTACHMENTS
    ap.add_argument("--csv", default=CSV_PATH, help="ruta del CSV con las URLs del origen")
    ap.add_argument("--sep", default=CSV_SEP, help="separador del CSV")
    ap.add_argument("--url-column", default=CSV_URL_COLUMN, help="columna con la URL")
    ap.add_argument("--all", action="store_true",
                    help="ignora el CSV y migra TODO el origen (export incremental)")
    ap.add_argument("--ticket", type=int, help="migrar solo este ID del origen")
    ap.add_argument("--limit", type=int, help="máximo de tickets a procesar")
    ap.add_argument("--start-date", help="YYYY-MM-DD (solo con --all)")
    ap.add_argument("--dry-run", action="store_true", help="arma el payload pero no crea en destino")
    ap.add_argument("--skip-attachments", action="store_true",
                    help="no descarga/sube adjuntos (útil para dry-run)")
    args = ap.parse_args()
    SKIP_ATTACHMENTS = args.skip_attachments or args.dry_run

    src = Zendesk(SRC, auth_header_origen)     # Basic
    dst = Zendesk(DST, auth_header_destino)    # Bearer (login())
    m = Mapper(src, dst)
    cp = load_checkpoint()

    if args.ticket:
        tickets = [src.get(f"/api/v2/tickets/{args.ticket}.json")["ticket"]]
    elif args.all:
        start = 0
        if args.start_date:
            start = int(datetime.strptime(args.start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
        tickets = iter_source_tickets(src, start)
    else:
        ids = ids_from_csv(args.csv, args.sep, args.url_column)
        ids = [i for i in ids if str(i) not in cp]      # salta los ya migrados
        if args.limit:
            ids = ids[:args.limit]
        tickets = iter_tickets_by_ids(src, ids)

    ok = fail = skipped = 0
    for i, t in enumerate(tickets, 1):
        if args.limit and i > args.limit:
            break
        sid = str(t["id"])
        if t["status"] == "deleted":
            continue
        if sid in cp:
            skipped += 1
            continue
        try:
            payload = build_payload(src, dst, m, t)
            if args.dry_run:
                print(json.dumps(payload, ensure_ascii=False, indent=2)[:3000])
                ok += 1
                continue
            r = dst.post("/api/v2/imports/tickets.json", json=payload)
            if r.ok:
                new_id = r.json()["ticket"]["id"]
                cp[sid] = new_id
                save_checkpoint(cp)
                ok += 1
                log.info("OK  %s -> %s", sid, new_id)
            else:
                raise RuntimeError(f"{r.status_code} {r.text[:500]}")
        except Exception as e:
            fail += 1
            log.error("FALLO ticket %s: %s", sid, e)
            with open(ERRORS_FILE, "a") as f:
                f.write(json.dumps({"src_id": t["id"], "error": str(e)}, ensure_ascii=False) + "\n")

    log.info("Terminado. OK=%s  FALLIDOS=%s  YA MIGRADOS=%s", ok, fail, skipped)


if __name__ == "__main__":
    main()